# Kapitel 5: Was hat die Maschine gelernt?

## Die Reise bis hierher

- **Kapitel 1** — Aus 3,6 Mio. echten Amazon-Bewertungen zogen wir 400.000
- **Kapitel 2** — Wir entdeckten die natürliche Verteilung und balancierten sie
- **Kapitel 3** — Wörter wurden zu TF-IDF-Vektoren
- **Kapitel 4** — Logistic Regression lernte, Emotionen zu erkennen

Jetzt erzählen wir die Geschichte visuell.

## 5.1 Setup

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Visualisierung") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df_clean = spark.read.parquet("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/cleaned_reviews.parquet")
df_preds = spark.read.parquet("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/predictions.parquet")

print(f"Bereinigte Daten: {df_clean.count():,}")
print(f"Vorhersagen:      {df_preds.count():,}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'font.size': 12, 'axes.titlesize': 14, 'axes.titleweight': 'bold'})

## 5.2 Frage 1: Wie sieht der balancierte Datensatz aus?

In [ ]:
from pyspark.sql.functions import col, count as spark_count

label_dist = df_clean.groupBy("label").agg(spark_count("*").alias("count")).orderBy("label").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#F09595", "#5DCAA5"]

bars = axes[0].bar(["Negativ (0)", "Positiv (1)"], label_dist["count"],
                   color=colors, edgecolor=["#A32D2D", "#0F6E56"])
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2., h, f'{int(h):,}', ha='center', va='bottom')
axes[0].set_title("Sentiment-Verteilung (nach Balancierung)")
axes[0].set_ylabel("Anzahl")

axes[1].pie(label_dist["count"], labels=["Negativ","Positiv"], colors=colors,
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title("Anteil")

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/01_sentiment_verteilung.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Frage 2: Schreiben zufriedene Kunden kürzer?

In [ ]:
from pyspark.sql.functions import length

df_len = df_clean.withColumn("text_length", length(col("text")))
pos_len = df_len.filter(col("label")==1).select("text_length").toPandas()
neg_len = df_len.filter(col("label")==0).select("text_length").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(neg_len["text_length"].dropna(), bins=50, alpha=0.6, color="#F09595", edgecolor="#A32D2D", label="Negativ", range=(0,3000))
ax.hist(pos_len["text_length"].dropna(), bins=50, alpha=0.6, color="#5DCAA5", edgecolor="#0F6E56", label="Positiv", range=(0,3000))
ax.set_xlabel("Textlänge (Zeichen)"); ax.set_ylabel("Anzahl")
ax.set_title("Textlängen nach Sentiment"); ax.legend()
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/02_textlaenge_sentiment.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Frage 3: Welche Wörter verraten die Stimmung?

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from pyspark.sql.functions import explode

tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered")
df_filt = remover.transform(tokenizer.transform(df_clean))
df_words = df_filt.select("label", explode("filtered").alias("word"))

top_pos = df_words.filter(col("label")==1).groupBy("word").agg(spark_count("*").alias("count")).orderBy(col("count").desc()).limit(20).toPandas()
top_neg = df_words.filter(col("label")==0).groupBy("word").agg(spark_count("*").alias("count")).orderBy(col("count").desc()).limit(20).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].barh(top_pos["word"][::-1], top_pos["count"][::-1], color="#5DCAA5", edgecolor="#0F6E56")
axes[0].set_title("Top 20 – POSITIV"); axes[0].set_xlabel("Häufigkeit")
axes[1].barh(top_neg["word"][::-1], top_neg["count"][::-1], color="#F09595", edgecolor="#A32D2D")
axes[1].set_title("Top 20 – NEGATIV"); axes[1].set_xlabel("Häufigkeit")
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/03_top_woerter.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.5 Word Cloud

In [ ]:
from wordcloud import WordCloud

pos_freq = dict(zip(top_pos["word"], top_pos["count"]))
neg_freq = dict(zip(top_neg["word"], top_neg["count"]))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
wc_pos = WordCloud(width=800, height=400, background_color="white", colormap="Greens", max_words=80).generate_from_frequencies(pos_freq)
axes[0].imshow(wc_pos, interpolation="bilinear"); axes[0].set_title("POSITIV", fontsize=16, fontweight='bold'); axes[0].axis("off")
wc_neg = WordCloud(width=800, height=400, background_color="white", colormap="Reds", max_words=80).generate_from_frequencies(neg_freq)
axes[1].imshow(wc_neg, interpolation="bilinear"); axes[1].set_title("NEGATIV", fontsize=16, fontweight='bold'); axes[1].axis("off")
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/04_wordcloud.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.6 Confusion Matrix

In [ ]:
cm_data = df_preds.groupBy("label","prediction").count().orderBy("label","prediction").toPandas()
cm = np.zeros((2,2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row["label"]), int(row["prediction"])] = int(row["count"])
cm_pct = cm / cm.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
labels = ["Negativ (0)", "Positiv (1)"]

for idx, (data, fmt, title) in enumerate([(cm, "{:,}", "absolut"), (cm_pct, "{:.1f}%", "prozentual")]):
    im = axes[idx].imshow(data, cmap="Blues")
    axes[idx].set_xticks([0,1]); axes[idx].set_yticks([0,1])
    axes[idx].set_xticklabels(labels); axes[idx].set_yticklabels(labels)
    axes[idx].set_xlabel("Vorhersage"); axes[idx].set_ylabel("Tatsächlich")
    axes[idx].set_title(f"Confusion Matrix ({title})")
    for i in range(2):
        for j in range(2):
            color = "white" if data[i,j] > data.max()/2 else "black"
            axes[idx].text(j, i, fmt.format(data[i,j]), ha="center", va="center", fontsize=16, fontweight="bold", color=color)

plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/05_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.7 ROC-Kurve

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
from sklearn.metrics import roc_curve, auc

extract_prob = udf(lambda prob: float(prob[1]), FloatType())
roc_pd = df_preds.withColumn("prob_positive", extract_prob(col("probability"))).select("label","prob_positive").toPandas()

fpr, tpr, _ = roc_curve(roc_pd["label"], roc_pd["prob_positive"])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color="#378ADD", lw=2.5, label=f"Logistic Regression (AUC = {roc_auc:.4f})")
ax.plot([0,1],[0,1], color="#B4B2A9", lw=1.5, linestyle="--", label="Zufall (AUC = 0.5)")
ax.fill_between(fpr, tpr, alpha=0.15, color="#85B7EB")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC-Kurve"); ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/06_roc_kurve.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.8 Wahrscheinlichkeits-Verteilung

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(roc_pd[roc_pd["label"]==0]["prob_positive"], bins=50, alpha=0.6, color="#F09595", edgecolor="#A32D2D", label="Negativ")
ax.hist(roc_pd[roc_pd["label"]==1]["prob_positive"], bins=50, alpha=0.6, color="#5DCAA5", edgecolor="#0F6E56", label="Positiv")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1.5, label="Schwellenwert")
ax.set_xlabel("Wahrscheinlichkeit (positiv)"); ax.set_ylabel("Anzahl")
ax.set_title("Verteilung der Vorhersage-Wahrscheinlichkeiten"); ax.legend()
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/08_wahrscheinlichkeiten.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.9 Metriken-Dashboard

In [ ]:
total_p = cm.sum()
acc = (cm[0,0]+cm[1,1])/total_p
prec_pos = cm[1,1]/(cm[0,1]+cm[1,1]) if (cm[0,1]+cm[1,1])>0 else 0
rec_pos = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
f1_v = 2*prec_pos*rec_pos/(prec_pos+rec_pos) if (prec_pos+rec_pos)>0 else 0

metrics = {'Accuracy': acc, 'AUC-ROC': roc_auc, 'Precision (pos)': prec_pos, 'Recall (pos)': rec_pos, 'F1 (pos)': f1_v}

fig, ax = plt.subplots(figsize=(8, 4))
names = list(metrics.keys()); values = list(metrics.values())
bars = ax.barh(names[::-1], values[::-1], color="#378ADD", height=0.5)
for bar in bars:
    w = bar.get_width()
    ax.text(w+0.01, bar.get_y()+bar.get_height()/2., f'{w:.4f}', ha='left', va='center', fontsize=11, fontweight='bold')
ax.set_xlim([0,1.15]); ax.set_title("Evaluations-Metriken")
ax.axvline(x=0.5, color="#B4B2A9", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/07_metriken_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()

## 5.10 Gespeicherte Dateien

In [ ]:
import os
output_dir = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output"
print("=" * 55)
print("  GESPEICHERTE DATEIEN")
print("=" * 55)
for f in sorted(os.listdir(output_dir)):
    fp = os.path.join(output_dir, f)
    if os.path.isfile(fp):
        print(f"  {f:45s} {os.path.getsize(fp)/(1024*1024):>6.2f} MB")
    else:
        print(f"  {f:45s} [Ordner]")
print("=" * 55)

## Fazit: Die Antwort auf unsere Frage

### Kann eine Maschine Emotionen lesen?

**Ja — mit Einschränkungen.**

Aus 3,6 Millionen echten Amazon-Bewertungen haben wir eine Stichprobe gezogen,
die natürliche Klassenverteilung analysiert, das Ungleichgewicht korrigiert
und mit TF-IDF + Logistic Regression eine Sentiment-Klassifikation aufgebaut.

### Was das Modell gelernt hat

- *great, love, best, well* → Signale für Zufriedenheit
- *money, bought, product, work* → Signale für Unzufriedenheit

### Der Unterschied zur vorgefertigten Test-Datei

Die `test.csv` auf Kaggle war bereits perfekt balanciert (50/50).
Wir haben bewusst die **echten Trainingsdaten** verwendet, um zu zeigen:
- Reale Daten sind selten perfekt
- Ungleichgewichte müssen erkannt und behandelt werden
- Die Pipeline funktioniert auch unter realistischen Bedingungen

### Mögliche Verbesserungen

| Ansatz | Erwartete Verbesserung |
|--------|----------------------|
| N-Gramme | Kontextbezug: *not good* ≠ *good* |
| Word2Vec | Semantische Ähnlichkeiten |
| Deep Learning (BERT) | Satzstruktur und Kontext |

---

*Vollständig mit Apache Spark und Python — von 3,6 Mio. Rohdaten zur Vorhersage.*